# Phase 6 — Online Evaluation and Production Monitoring

Databricks AI Evals Tutorial | Phase 6 of 10

Everything so far has been **offline**: a fixed dataset, scored on demand, producing a
ship/no-ship decision. Offline evaluation has one structural limit, and it's the same one
Phase 4 worked around — **it can only ever catch what your dataset contains.**

Online evaluation scores live traffic continuously. It is how you find out what neither
your curated set nor your mined set had yet.

The two are not competing; they do different jobs:

| | Offline (Phases 2-5) | Online (this phase) |
|---|---|---|
| Input | A fixed dataset you control | Whatever users actually send |
| Ground truth | Yes — you wrote it | **None** |
| Coverage | 100% of the dataset | A *sample* of traffic |
| Answers | "Did this change break anything?" | "What is happening that we never tested?" |
| Runs | On demand, before a release | Continuously, after it |

> **This is the first phase that genuinely requires a Databricks workspace.** Phases 0-5
> run locally against SQLite. Unity Catalog trace ingestion and registered monitoring
> scorers are workspace features with no local equivalent.

## The structural fact that decides everything: production has no ground truth

Nobody wrote `expected_facts` for the question a customer asked thirty seconds ago, and
nobody ever will. Every scorer that reads `expectations` is therefore **offline-only** — not
because it's badly written, but because its input doesn't exist in production.

This bites in a specific, easy-to-miss way. `tool_call_correctness` from Phase 3 is one of
the most useful scorers in this whole track, and it **cannot run online at all** — it reads
`expectations["expects_tool_call"]`, so against live traffic it would return `skip` on every
single trace while appearing perfectly healthy.

Check this before registering anything, not after.

In [ ]:
# ============ WHICH SCORERS TRANSFER TO PRODUCTION ============
import monitoring as M

PHASE_2_3_SCORERS = [
    "Safety", "RelevanceToQuery", "RetrievalGroundedness", "Guidelines",
    "no_account_leakage", "response_word_count",
    "Correctness", "ExpectationsGuidelines", "tool_call_correctness",
]

print(f"{'SCORER':<26}{'ONLINE?':<10}WHY")
print("-" * 100)
for name in PHASE_2_3_SCORERS:
    can_run, reason = M.classify_scorer(name)
    tag = "yes" if can_run else "NO"
    print(f"{name:<26}{tag:<10}{reason[:64]}")

print()
print("The test is mechanical: does the scorer read `expectations`?")
print("If yes, it is offline-only, however good it is.")


## Prerequisites

Before any code runs, you need four things in place.

**1. A SQL warehouse.** Find its ID with the Databricks CLI:

```bash
databricks warehouses list --profile <YOUR_PROFILE>
```

**2. A Unity Catalog catalog and schema** you can create tables in.

**3. Explicit grants.** `ALL_PRIVILEGES` is **not sufficient** — the trace tables need
`MODIFY` and `SELECT` granted by name:

```sql
GRANT USE_CATALOG ON CATALOG <catalog> TO `you@example.com`;
GRANT USE_SCHEMA  ON SCHEMA  <catalog>.<schema> TO `you@example.com`;

GRANT MODIFY, SELECT ON TABLE <catalog>.<schema>.mlflow_experiment_trace_otel_logs    TO `you@example.com`;
GRANT MODIFY, SELECT ON TABLE <catalog>.<schema>.mlflow_experiment_trace_otel_spans   TO `you@example.com`;
GRANT MODIFY, SELECT ON TABLE <catalog>.<schema>.mlflow_experiment_trace_otel_metrics TO `you@example.com`;
```

You'll also need `CAN USE` on the warehouse and `CAN EDIT` on the experiment.

**4. `mlflow[databricks]>=3.9.0`.** UC trace ingestion does not exist below 3.9 — this repo
pins 3.16, so that's satisfied.

Two caveats worth knowing before you invest setup time: UC trace ingestion is in **Beta and
limited to `us-east-1` / `us-west-2`**, and ingestion is capped around **100 traces/second
per workspace**.

## A trap that will cost you your earlier work

**Linking a UC schema to an experiment hides that experiment's pre-existing MLflow-stored
traces.** Unlinking restores them, but while linked they're invisible.

So do **not** link `telcoassist-evals` — that's where Phases 1-5 wrote everything. This
notebook uses a separate `/Shared/telcoassist-production` experiment, which is also the
right shape in general: evaluation runs and production traffic are different things with
different retention, different access, and different volume.

In [ ]:
# ============ CONFIGURATION -- FILL THESE IN ============
import os

import mlflow

# --- your workspace values --------------------------------------------------
CATALOG_NAME = "<YOUR_CATALOG>"
SCHEMA_NAME = "<YOUR_SCHEMA>"
SQL_WAREHOUSE_ID = "<YOUR_SQL_WAREHOUSE_ID>"

# Deliberately NOT the evaluation experiment -- linking would hide Phases 1-5 traces.
PROD_EXPERIMENT = "/Shared/telcoassist-production"

placeholders = [v for v in (CATALOG_NAME, SCHEMA_NAME, SQL_WAREHOUSE_ID) if v.startswith("<")]
if placeholders:
    raise ValueError(
        "Fill in CATALOG_NAME, SCHEMA_NAME and SQL_WAREHOUSE_ID before running this "
        f"notebook. Still unset: {placeholders}. "
        "Find the warehouse id with: databricks warehouses list --profile <PROFILE>"
    )

mlflow.set_tracking_uri("databricks")
os.environ["TELCOASSIST_PROVIDER"] = "databricks"

# Must be set BEFORE linking the UC schema, or the link call fails.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = SQL_WAREHOUSE_ID

print(f"catalog.schema : {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"warehouse      : {SQL_WAREHOUSE_ID}")
print(f"experiment     : {PROD_EXPERIMENT}")


In [ ]:
# ============ LINK THE UC SCHEMA TO A FRESH EXPERIMENT ============
from mlflow.entities import UCSchemaLocation
from mlflow.tracing.enablement import set_experiment_trace_location

if experiment := mlflow.get_experiment_by_name(PROD_EXPERIMENT):
    prod_experiment_id = experiment.experiment_id
    print(f"using existing experiment (id {prod_experiment_id})")
else:
    prod_experiment_id = mlflow.create_experiment(name=PROD_EXPERIMENT)
    print(f"created experiment (id {prod_experiment_id})")

set_experiment_trace_location(
    location=UCSchemaLocation(catalog_name=CATALOG_NAME, schema_name=SCHEMA_NAME),
    experiment_id=prod_experiment_id,
)

print()
print("Three tables are created automatically:")
for suffix in ("otel_spans", "otel_logs", "otel_metrics"):
    print(f"  {CATALOG_NAME}.{SCHEMA_NAME}.mlflow_experiment_trace_{suffix}")


In [ ]:
# ============ POINT TRACING AT UNITY CATALOG ============
mlflow.set_experiment(PROD_EXPERIMENT)

mlflow.tracing.set_destination(
    destination=UCSchemaLocation(catalog_name=CATALOG_NAME, schema_name=SCHEMA_NAME)
)

# Equivalent, for deployment configs where you set env vars rather than call an API.
# Note the format is dot-separated catalog.schema -- not a slash, not the catalog alone.
#     os.environ["MLFLOW_TRACING_DESTINATION"] = f"{CATALOG_NAME}.{SCHEMA_NAME}"

mlflow.langchain.autolog()
print("tracing destination set -- traces from this process now land in Unity Catalog")


## Sampling, part 1: what it costs to score everything

Scoring 100% of production traffic with LLM judges is frequently more expensive than
serving the traffic was. That's the whole reason `ScorerSamplingConfig` exists.

The split follows directly from Phase 3's cost hierarchy: **deterministic scorers are free,
so never sample them; judges cost money per call, so sample them.**

In [ ]:
# ============ COST OF MONITORING ============
DAILY_TRACES = 2000        # requests/day this agent serves
JUDGE_SCORERS = 4          # LLM-judge scorers you want running

print(f"{DAILY_TRACES:,} traces/day, {JUDGE_SCORERS} judge scorers")
print()
print(f"{'SAMPLE':<10}{'JUDGE CALLS/DAY':>18}{'COST/DAY':>12}{'COST/YEAR':>14}")
print("-" * 56)
for rate in (1.0, 0.50, 0.25, 0.10, 0.05):
    c = M.monitoring_cost(DAILY_TRACES, JUDGE_SCORERS, rate)
    print(f"{rate:<10.0%}{c['judge_calls_per_day']:>18,.0f}"
          f"{c['cost_per_day']:>12,.2f}{c['cost_per_year']:>14,.0f}")

print()
print("Deterministic scorers are absent from this table on purpose: they cost nothing,")
print("which is exactly why they should run at 100% while judges are sampled.")


## Sampling, part 2: what a sample can actually resolve

This is where production monitoring most often goes wrong, and it has nothing to do with
MLflow.

A sampled pass rate is an **estimate**, and estimates have error bars. Sample 5% of traffic,
watch a metric move from 0.95 to 0.93, page someone — when the confidence interval on that
sample was ±0.045 and the move was indistinguishable from noise. Alerts configured this way
train people to ignore alerts.

So before setting a threshold, work out what the sample can resolve.

In [ ]:
# ============ WHAT CAN A 5% SAMPLE SEE? ============
BASELINE = 0.95
SAMPLE_RATE = 0.05

scored = int(DAILY_TRACES * SAMPLE_RATE)
halfwidth = M.interval_halfwidth(BASELINE, scored)
low, high = M.wilson_interval(BASELINE * scored, scored)

print(f"{DAILY_TRACES:,} traces/day at {SAMPLE_RATE:.0%} -> {scored} traces scored")
print(f"observed rate {BASELINE:.2f} -> 95% interval [{low:.3f}, {high:.3f}]")
print(f"resolution: +/-{halfwidth:.3f}  ({halfwidth * 100:.1f} percentage points)")
print()
for delta in (0.10, 0.05, 0.02, 0.01):
    verdict = "detectable" if delta > halfwidth else "INVISIBLE -- would be noise"
    print(f"  a {delta * 100:>4.0f}-point drop is {verdict}")


In [ ]:
# ============ HOW MUCH SAMPLING WOULD IT TAKE? ============
print(f"{'DELTA TO DETECT':<20}{'SAMPLES NEEDED':>16}{'RATE @ ' + str(DAILY_TRACES) + '/day':>20}")
print("-" * 58)
for delta in (0.10, 0.05, 0.02, 0.01):
    needed = M.min_samples_to_detect(BASELINE, delta)
    rate = M.required_sample_rate(DAILY_TRACES, BASELINE, delta)
    rate_text = f"{rate:.0%}" if rate else "not achievable in a day"
    print(f"{delta * 100:>6.0f} points{'':<8}{needed if needed else '-':>16}{rate_text:>20}")

print()
print("Two practical conclusions:")
print("  1. Pick the sample rate from the regression size you need to catch,")
print("     not from a round number that felt affordable.")
print("  2. If a delta isn't achievable even at 100%, don't alert on it at all --")
print("     catch it offline instead, where you score every row of a fixed dataset.")


That second point is the honest division of labour between this phase and Phase 5. A small
regression is **cheap** to detect offline, because you score 100% of a fixed dataset and the
comparison is paired. The same regression can be **unaffordable** to detect online. Online
monitoring is for discovering problems you had no test for — not for precisely measuring
ones you did.

## Register and start the scorers

Two calls, and **both are required**. `register()` creates the scorer; `start()` begins
evaluating. Registering alone leaves a scorer that exists and does nothing — a silent
failure that looks exactly like healthy monitoring.

Note the sample rates below follow the rule rather than a habit: deterministic at 100%,
judges sampled, and safety judged more heavily than tone because the cost of missing it is
higher.

In [ ]:
# ============ REGISTER + START ============
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
    RetrievalGroundedness,
    Safety,
    ScorerSamplingConfig,
)
from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id

import scorers as S

set_databricks_monitoring_sql_warehouse_id(
    warehouse_id=SQL_WAREHOUSE_ID,
    experiment_id=prod_experiment_id,
)

# --- deterministic: free, so no reason to sample -----------------------------
leakage_monitor = S.no_account_leakage.register(name="prod_account_leakage")
leakage_monitor = leakage_monitor.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
print("prod_account_leakage   started at 100% (deterministic, free)")

# --- judges: sampled, weighted by consequence --------------------------------
safety_monitor = Safety().register(name="prod_safety")
safety_monitor = safety_monitor.start(sampling_config=ScorerSamplingConfig(sample_rate=0.30))
print("prod_safety            started at  30% (judge)")

grounded_monitor = RetrievalGroundedness().register(name="prod_groundedness")
grounded_monitor = grounded_monitor.start(sampling_config=ScorerSamplingConfig(sample_rate=0.20))
print("prod_groundedness      started at  20% (judge)")

# The abstention scorer Phase 4's mined traffic motivated. It is reference-free, so unlike
# tool_call_correctness it transfers to production unchanged.
abstention_monitor = Guidelines(
    name="abstains_when_unsupported",
    guidelines=(
        "If the support articles provided to the agent do not contain the information "
        "needed to answer the question, the response must say the information is not "
        "available and offer to connect the customer with a human agent. It must not state "
        "specific plan names, prices, or eligibility rules that are absent from the "
        "articles. If the articles do cover the question, this guideline is automatically "
        "satisfied."
    ),
).register(name="prod_abstention")
abstention_monitor = abstention_monitor.start(sampling_config=ScorerSamplingConfig(sample_rate=0.20))
print("prod_abstention        started at  20% (judge)")


In [ ]:
# ============ VERIFY WHAT IS ACTUALLY RUNNING ============
from mlflow.genai.scorers import list_scorers

print(f"{'SCORER':<26}{'SAMPLE RATE':>14}")
print("-" * 42)
for s in list_scorers():
    config = getattr(s, "sampling_config", None)
    rate = f"{config.sample_rate:.0%}" if config else "not started"
    print(f"{s.name:<26}{rate:>14}")

print()
print("A scorer showing 'not started' was registered but never started -- it exists,")
print("it evaluates nothing, and nothing about the UI will tell you so.")


## Generate traffic and let the monitors see it

Reusing Phase 4's traffic simulator: same skewed mix, same long tail containing questions
the knowledge base cannot answer.

Scoring is **asynchronous**. Traces land immediately; scorer results appear shortly after,
for the sampled subset only.

In [ ]:
# ============ SEND PRODUCTION TRAFFIC ============
import agent
import traffic as T

REQUESTS = T.simulate_traffic(n=40, seed=21)

for i, (query, customer_id, note) in enumerate(REQUESTS, 1):
    agent.answer(query, customer_id=customer_id)
    if i % 10 == 0:
        print(f"  sent {i}/{len(REQUESTS)} requests")

print()
print(f"{len(REQUESTS)} traces written to {CATALOG_NAME}.{SCHEMA_NAME}")
print("Scorers evaluate asynchronously on their sampled subset -- results follow shortly.")


## Query the traces as data

This is the part that has no offline equivalent. Traces in Unity Catalog are **Delta
tables**, so production behaviour is queryable with SQL, joinable against other tables, and
usable as a dashboard source.

The `..._otel_spans` table holds one row per span. Root spans have `parent_span_id IS NULL`,
which is how you count *traces* rather than spans.

In [ ]:
# ============ SQL OVER THE TRACE TABLES ============
SPANS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.mlflow_experiment_trace_otel_spans"

VOLUME_SQL = f"""
SELECT DATE(timestamp) AS day,
       COUNT(DISTINCT trace_id) AS traces
FROM {SPANS_TABLE}
WHERE parent_span_id IS NULL          -- root spans only = one row per trace
GROUP BY DATE(timestamp)
ORDER BY day DESC
"""

LATENCY_SQL = f"""
SELECT trace_id,
       name,
       (end_time_unix_nano - start_time_unix_nano) / 1e9 AS duration_seconds
FROM {SPANS_TABLE}
WHERE parent_span_id IS NULL
ORDER BY duration_seconds DESC
LIMIT 20
"""

# Per-span-type latency: which stage of the agent actually costs the time?
STAGE_SQL = f"""
SELECT name,
       COUNT(*) AS calls,
       ROUND(AVG((end_time_unix_nano - start_time_unix_nano) / 1e6), 1) AS avg_ms,
       ROUND(PERCENTILE(( end_time_unix_nano - start_time_unix_nano) / 1e6, 0.95), 1) AS p95_ms
FROM {SPANS_TABLE}
GROUP BY name
ORDER BY avg_ms DESC
"""

ERROR_SQL = f"""
SELECT name,
       COUNT(*) AS total,
       SUM(CASE WHEN status_code = 'ERROR' THEN 1 ELSE 0 END) AS errors,
       ROUND(SUM(CASE WHEN status_code = 'ERROR' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS error_pct
FROM {SPANS_TABLE}
GROUP BY name
HAVING COUNT(*) > 5
ORDER BY error_pct DESC
"""

for label, sql in [("VOLUME", VOLUME_SQL), ("SLOWEST TRACES", LATENCY_SQL),
                   ("LATENCY BY STAGE", STAGE_SQL), ("ERRORS BY STAGE", ERROR_SQL)]:
    print(f"--- {label} ---")
    print(sql.strip())
    print()


In [ ]:
# ============ RUN ONE OF THEM ============
# Via Databricks Connect. Outside a Databricks notebook this needs `databricks-connect`
# configured against your profile; inside one, `spark` already exists.
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.remote(serverless=True).getOrCreate()

print("latency by stage:")
spark.sql(STAGE_SQL).show(truncate=False)


Read that stage table against Phase 1's span tree. `retrieve_kb`, the model calls, and
`lookup_account` each appear as their own row — so "the agent got slower" becomes "retrieval
p95 doubled", which is a different and far more actionable statement.

This is the payoff for Phase 1 instrumenting spans explicitly rather than leaving trace
shape to framework autologging.

## Managing monitors over time

Sample rates are not set once. The usual pattern: raise the rate temporarily while
investigating something, then lower it again; stop a monitor during a known-noisy migration;
delete one whose scorer you've replaced.

`stop()` keeps the registration so you can restart it later. `delete_scorer()` removes it
entirely.

In [ ]:
# ============ UPDATE / STOP / DELETE ============
from mlflow.genai.scorers import delete_scorer, get_scorer

# Investigating a groundedness complaint: raise resolution temporarily.
grounded = get_scorer(name="prod_groundedness")
grounded = grounded.update(sampling_config=ScorerSamplingConfig(sample_rate=0.60))
print(f"prod_groundedness raised to {grounded.sampling_config.sample_rate:.0%} for investigation")

# ... investigation over, return to the steady-state rate.
grounded = grounded.update(sampling_config=ScorerSamplingConfig(sample_rate=0.20))
print(f"prod_groundedness returned to {grounded.sampling_config.sample_rate:.0%}")

# Pause without losing the registration.
abstention = get_scorer(name="prod_abstention")
abstention = abstention.stop()
print("prod_abstention stopped (registration kept -- restart with .start())")

# Restart it.
abstention = abstention.start(sampling_config=ScorerSamplingConfig(sample_rate=0.20))
print("prod_abstention restarted at 20%")

# Permanent removal, for when a scorer has been superseded:
#     delete_scorer(name="prod_abstention")


## Closing the loop

Online and offline evaluation are two halves of one system, and the handoff between them is
the thing worth internalising:

```
            production traffic
                    |
                    v
   [Phase 6]  online monitoring  ->  a sampled judge flags something
                    |                  no ground truth, wide error bars,
                    |                  enough to say "look here"
                    v
   [Phase 4]  mine those traces  ->  targeted selection, tagged, human labels
                    |                  with AssessmentSource provenance
                    v
   [Phase 2]  add to curated set ->  now a fixed, fully-scored test case
                    |
                    v
   [Phase 5]  promotion gate     ->  every future release is checked against it
                                       100% coverage, paired comparison, cheap
```

Online evaluation **discovers**; offline evaluation **prevents recurrence**. A finding that
stops at the monitoring dashboard protects you once. The same finding carried through to a
gated test protects every release after it.

That is also the answer to the interview question in `../Sample_Questions/` Case #8 — *"the
offline benchmark improved but the customer says quality declined."* If the only measurement
is offline, the benchmark is whatever you thought to write down. Online monitoring is what
tells you the benchmark has drifted from reality.

## Key takeaways

- **Production has no ground truth, and that decides which scorers are even eligible.** Any
  scorer reading `expectations` is offline-only — including `tool_call_correctness`, which
  would silently return `skip` on every live trace while looking perfectly healthy.
- **`register()` and `start()` are both required.** A registered-but-unstarted scorer
  evaluates nothing and looks identical to a working one.
- **Sample judges, never sample deterministic scorers.** The first cost money per call; the
  second are free. Phase 3's cost hierarchy becomes a sampling policy here.
- **Pick sample rates from the regression size you need to detect.** At 2,000 traces/day, a
  5% sample resolves about ±4.5 points — so a 2-point drop is invisible, and an alert set
  there fires on noise. Detecting 2 points needs roughly 30% sampling.
- **Some deltas aren't worth chasing online at all.** Catch small regressions offline,
  where you score 100% of a fixed dataset and the comparison is paired.
- **Linking a UC schema hides that experiment's existing traces** — use a separate
  experiment for production rather than the one holding your evaluation history.
- **Traces in UC are Delta tables**, so production behaviour is SQL-queryable and joinable.
  Per-span rows turn "the agent got slower" into "retrieval p95 doubled".
- **Online discovers, offline prevents recurrence.** Carry every finding through the loop
  into a gated test, or it only ever helps once.

**Next: Phase 7 (bonus) — the judges doing all this scoring are themselves unvalidated.
Aligning one to human expert judgement is what makes any of these numbers trustworthy.**